# Modèle de régimes macro pour une allocation multi-actifs

Ce notebook a deux objectifs complémentaires.

1. **Classifier chaque mois dans un régime macroéconomique** à partir des indicateurs contenus dans `régime_macro.xlsx`.
2. **Tester une vraie approche de classification supervisée de type machine learning**, entraînée sur l'historique **du début de l'échantillon jusqu'à 2014-12**, puis appliquée hors échantillon sur **2015-01 à 2026-02**.

Le cas d'usage est une stratégie d'investissement multi-actifs répartie entre quatre classes d'actifs :

- actions (`equity`)
- taux / crédit (`rates_credit`)
- devises (`fx`)
- matières premières (`commodities`)

Le portefeuille de départ est égal pondéré à 25 % par classe d'actifs. L'objectif du projet est de montrer qu'une allocation dynamique fondée sur les régimes macro permet d'améliorer cette allocation naïve.

**Point méthodologique important.**  
Ici, on ne cherche pas à prédire le régime à `t+1`. On cherche à estimer, comme dans un problème classique de machine learning supervisé, quelle combinaison d'indicateurs contemporains explique le mieux **l'état du monde au mois considéré**. On entraîne donc un classifieur sur 1990–2019, puis on lui demande de **reconnaître** le régime des mois 2020–2026 à partir des variables observées sur ces mêmes mois. La validation économique consiste ensuite à comparer cette chronologie prédite à la réalité macroéconomique observée.

## Cadre économique et choix méthodologiques

Dans ce projet, l’objectif est de construire un modèle d’allocation dynamique multi-actifs basé sur l’identification de régimes macroéconomiques. Le portefeuille de référence est initialement construit avec une pondération égale entre quatre classes d’actifs : actions (equity), taux/crédit (rates/credit), devises (FX) et matières premières (commodities). Cette allocation naïve sert de benchmark. L’enjeu consiste à améliorer cette allocation en adaptant les pondérations en fonction de l’état du cycle économique.

Pour cela, nous construisons un indicateur macro synthétique permettant de classifier chaque date de l’échantillon dans un régime économique. Cet indicateur repose sur trois dimensions fondamentales de l’environnement macroéconomique : la croissance, l’inflation et le stress financier. Cette structuration est cohérente avec la logique du fichier source, qui organise explicitement les variables selon ces trois blocs.

Le bloc de croissance regroupe des indicateurs avancés et contemporains de l’activité économique. Le PMI global mesure directement l’expansion ou la contraction du secteur manufacturier. Le CLI de l’OCDE fournit une information avancée sur le cycle. Les pentes de courbe (États-Unis, Allemagne, Royaume-Uni) capturent les anticipations de croissance via la structure par terme des taux. Enfin, le Citi Economic Surprise Index reflète l’écart entre les données publiées et les attentes du marché, ce qui permet de capter les révisions de trajectoire économique.

Le bloc d’inflation repose sur des indicateurs de prix réalisés et anticipés. Le CPI YoY mesure l’inflation observée. Les breakevens d’inflation, aux États-Unis et en Europe, capturent les anticipations d’inflation intégrées dans les marchés obligataires. Ces variables permettent de distinguer les phases de désinflation, de stabilité des prix et de pressions inflationnistes.

Le bloc de stress financier regroupe des indicateurs de volatilité et de conditions de financement. Le VIX et le MOVE mesurent respectivement la volatilité implicite des marchés actions et obligataires. Les indices de crédit (CDX et iTraxx) capturent le risque de défaut perçu. Le Goldman Sachs Financial Conditions Index synthétise l’état global des conditions financières. Ce bloc est essentiel pour identifier les épisodes de crise ou de resserrement financier.

À partir de ces trois blocs, nous construisons des scores synthétiques de croissance, d’inflation et de stress. Chaque variable est transformée en signal économiquement interprétable, soit via un seuil naturel, soit via une normalisation sur historique roulant. Les signaux sont ensuite agrégés en scores. Une attention particulière est portée au traitement des données manquantes : une absence de donnée ne doit jamais être interprétée comme un signal négatif.

Le modèle a ensuite deux étages distincts.

Le premier étage est **économique** : il sert à construire une étiquette de régime à partir des trois scores de croissance, d’inflation et de stress. Cette étape fournit une chronologie de référence du cycle.

Le deuxième étage est **statistique** : il consiste à entraîner un modèle de classification supervisée afin d’apprendre quelles combinaisons d’indicateurs expliquent le mieux les régimes observés sur l’échantillon d’entraînement. L’idée n’est donc pas de prédire `t+1`, mais d’identifier, comme dans un problème standard de machine learning, la combinaison de variables qui discrimine le mieux les différents états du monde.

L’échantillon est découpé de la façon suivante : la période allant du début de l’historique jusqu’à **2019-12** sert à l’apprentissage et à la sélection de modèle ; la période **2020-01 à 2026-02** sert de test hors échantillon. Sur cette seconde période, nous comparons la chronologie prédite à la fois aux régimes reconstruits à partir des données et à la réalité macroéconomique documentée par les institutions économiques internationales.

Une fois les régimes prédits, ils peuvent être traduits en pondérations de portefeuille. Chaque régime est associé à une allocation spécifique entre les quatre classes d’actifs. Par exemple, en régime de crise, le portefeuille est orienté vers les actifs défensifs comme les obligations et réduit son exposition aux actions. En régime de reflation, l’exposition aux matières premières et aux actions est augmentée. En régime de Goldilocks, les actifs risqués sont favorisés. En régime de stagflation, l’allocation privilégie les actifs réels et limite l’exposition aux actifs sensibles aux taux.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import itertools
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import TimeSeriesSplit

DATA_PATH = Path("régime_macro.xlsx")
TRAIN_END = "2014-12-31"
TEST_START = "2015-01-01"
RANDOM_STATE = 42


## 1. Charger et nettoyer les données

Le fichier Excel stocke les tickers Bloomberg sur la première ligne de la feuille `Data`, puis les données historiques en dessous.
Les valeurs manquantes Bloomberg comme `#N/A N/A` doivent rester manquantes. Elles ne doivent **jamais** être transformées en signal baissier par défaut.



In [3]:

def load_macro_data(xlsx_path: str | Path, sheet_name: str = "Data") -> pd.DataFrame:
    raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)
    cols = list(raw.iloc[0])
    cols[0] = "date"

    df = raw.iloc[3:].copy()
    df.columns = cols
    df = df[df["date"].notna()].copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[df["date"].notna()].copy()
    df = df.replace(["#N/A N/A", "#VALUE!", "#REF!", "#DIV/0!"], np.nan)

    for c in df.columns:
        if c != "date":
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.set_index("date").sort_index()
    return df

df = load_macro_data(DATA_PATH)
df.head()


,VIX Index,MOVE Index,ITRX EUR CDSI GEN 5Y Corp,CDX IG CDSI GEN 5Y Corp,GSGLFCI Index,USGGBE10 Index,FRGGBE10 Index,DEGGBE10 Index,GILGBE10 Index,SKGGBE10 Index,...,EUGPDE Index,WGDPITAL Index,WGDPSWED Index,CPI YOY Index,NAPMPMI Index,CESIUSD Index,OEUSKLAC Index,US Slope 10-2,EU (Ger) 10-2 Slope,UK Slope 10-2
date,,,,,,,,,,,,,,,,,,,,,
1990-01-31,25.36,105.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,5.2,47.2,NaN,99.8034,0.164,NaN,NaN
1990-02-28,21.99,105.72,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,5.3,49.1,NaN,99.8335,0.093,NaN,NaN
1990-03-30,19.73,103.89,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,5.2,49.9,NaN,99.8325,0.006,NaN,NaN
1990-04-30,19.52,104.63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,4.7,50.0,NaN,99.7572,0.078,NaN,NaN
1990-05-31,17.37,94.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,4.4,49.5,NaN,99.5861,0.117,NaN,NaN


In [4]:

availability = pd.DataFrame({
    "first_valid": df.apply(lambda s: s.first_valid_index()),
    "last_valid": df.apply(lambda s: s.last_valid_index()),
    "n_obs": df.notna().sum(),
    "coverage_pct": (df.notna().mean() * 100).round(1)
}).sort_values("first_valid")

availability


,first_valid,last_valid,n_obs,coverage_pct
VIX Index,1990-01-31,2026-02-27,434,100.0
MOVE Index,1990-01-31,2026-02-27,434,100.0
US Slope 10-2,1990-01-31,2026-02-27,434,100.0
OEUSKLAC Index,1990-01-31,2026-02-27,434,100.0
NAPMPMI Index,1990-01-31,2026-02-27,434,100.0
CPI YOY Index,1990-01-31,2026-02-27,434,100.0
FRGDP Index,1990-03-30,2026-02-27,432,99.5
EU (Ger) 10-2 Slope,1990-09-28,2026-02-27,426,98.2
WGDPITAL Index,1990-12-31,2026-02-27,423,97.5
WGDPSWED Index,1990-12-31,2026-02-27,423,97.5



## 2. Construire des variables économiquement interprétables

Nous évitons d'utiliser les niveaux bruts lorsqu'ils sont difficiles à comparer sur des horizons longs.
À la place, nous utilisons un petit ensemble de transformations cohérentes économiquement :

- les niveaux quand un seuil naturel existe : PMI > 50, pente de courbe > 0 ;
- les écarts à 12 mois ou à une moyenne mobile lorsque la dynamique est plus importante que le niveau ;
- les indicateurs de stress sont conservés séparément de l'inflation.



In [5]:

def zscore_rolling(s: pd.Series, window: int = 60, min_periods: int = 24) -> pd.Series:
    mu = s.rolling(window, min_periods=min_periods).mean()
    sd = s.rolling(window, min_periods=min_periods).std()
    return (s - mu) / sd

features = pd.DataFrame(index=df.index)

# Bloc croissance
features["pmi_level"] = df["NAPMPMI Index"] - 50
features["cli_gap_12m"] = df["OEUSKLAC Index"] - df["OEUSKLAC Index"].rolling(12, min_periods=6).mean()
features["slope_us"] = df["US Slope 10-2"]
features["cesi_us"] = df["CESIUSD Index"]

# Bloc inflation
features["cpi_yoy"] = df["CPI YOY Index"]
features["breakeven_us"] = df["USGGBE10 Index"]

# Stress / financial conditions
features["vix"] = df["VIX Index"]
features["move"] = df["MOVE Index"]
features["fci_us"] = df["GSGLFCI Index"]
features["cdx_ig"] = df["CDX IG CDSI GEN 5Y Corp"]
features["itraxx_eur"] = df["ITRX EUR CDSI GEN 5Y Corp"]

# Optional European inflation proxy: GDP-weighted breakeven
gdp_cols = ["FRGDP Index", "EUGPDE Index", "WGDPITAL Index", "WGDPSWED Index"]
be_cols = ["FRGGBE10 Index", "DEGGBE10 Index", "GILGBE10 Index", "SKGGBE10 Index"]

gdp = df[gdp_cols].copy()
be = df[be_cols].copy()
gdp.columns = ["FR", "DE", "IT", "SE"]
be.columns = ["FR", "DE", "IT", "SE"]

weights = gdp.div(gdp.sum(axis=1), axis=0)
features["breakeven_eu_proxy"] = (be * weights).sum(axis=1, min_count=1)

features.head()



,pmi_level,cli_gap_12m,slope_us,cesi_us,cpi_yoy,breakeven_us,vix,move,fci_us,cdx_ig,itraxx_eur,breakeven_eu_proxy
date,,,,,,,,,,,,
1990-01-31,-2.8,NaN,0.164,NaN,5.2,NaN,25.36,105.50,NaN,NaN,NaN,NaN
1990-02-28,-0.9,NaN,0.093,NaN,5.3,NaN,21.99,105.72,NaN,NaN,NaN,NaN
1990-03-30,-0.1,NaN,0.006,NaN,5.2,NaN,19.73,103.89,NaN,NaN,NaN,NaN
1990-04-30,0.0,NaN,0.078,NaN,4.7,NaN,19.52,104.63,NaN,NaN,NaN,NaN
1990-05-31,-0.5,NaN,0.117,NaN,4.4,NaN,17.37,94.70,NaN,NaN,NaN,NaN



## 3. Construire les scores composites de régime

Nous créons trois composites standardisés.

Un score élevé signifie :

- **growth_score** : activité plus solide ;
- **inflation_score** : pression inflationniste plus forte ;
- **stress_score** : marchés plus tendus ou plus fragiles.



In [6]:

scores = pd.DataFrame(index=features.index)

growth_components = pd.DataFrame(index=features.index)
growth_components["pmi"] = zscore_rolling(features["pmi_level"])
growth_components["cli"] = zscore_rolling(features["cli_gap_12m"])
growth_components["slope"] = zscore_rolling(features["slope_us"])
growth_components["cesi"] = zscore_rolling(features["cesi_us"])

infl_components = pd.DataFrame(index=features.index)
infl_components["cpi"] = zscore_rolling(features["cpi_yoy"])
infl_components["breakeven_us"] = zscore_rolling(features["breakeven_us"])
infl_components["breakeven_eu"] = zscore_rolling(features["breakeven_eu_proxy"])

stress_components = pd.DataFrame(index=features.index)
stress_components["vix"] = zscore_rolling(features["vix"])
stress_components["move"] = zscore_rolling(features["move"])
stress_components["fci"] = zscore_rolling(features["fci_us"])
stress_components["cdx"] = zscore_rolling(features["cdx_ig"])
stress_components["itraxx"] = zscore_rolling(features["itraxx_eur"])

scores["growth_score"] = growth_components.mean(axis=1)
scores["inflation_score"] = infl_components.mean(axis=1)
scores["stress_score"] = stress_components.mean(axis=1)

scores.head()



,growth_score,inflation_score,stress_score
date,,,
1990-01-31,NaN,NaN,NaN
1990-02-28,NaN,NaN,NaN
1990-03-30,NaN,NaN,NaN
1990-04-30,NaN,NaN,NaN
1990-05-31,NaN,NaN,NaN



## 4. Définir les régimes économiques

Nous utilisons des règles explicites.
Les seuils pourront ensuite être optimisés ou calibrés uniquement sur l'échantillon d'entraînement.

Choix actuel :

- croissance positive si `growth_score > 0` ;
- inflation positive si `inflation_score > 0` ;
- stress élevé si `stress_score > 0.5`.

Ensuite :

- croissance forte + inflation contenue + stress faible → `goldilocks` ;
- croissance forte + inflation forte + stress faible → `reflation` ;
- croissance faible + inflation forte → `stagflation` ;
- stress élevé ou croissance faible avec inflation faible → `crisis` ;
- cas intermédiaires → `slowdown`.



In [7]:

def classify_regime_row(row):
    g = row["growth_score"]
    i = row["inflation_score"]
    s = row["stress_score"]

    if pd.isna(g) or pd.isna(i):
        return np.nan

    g_pos = g > 0
    i_pos = i > 0
    stress_high = (s > 0.5) if pd.notna(s) else False

    if g_pos and (not i_pos) and (not stress_high):
        return "goldilocks"
    if g_pos and i_pos and (not stress_high):
        return "reflation"
    if (not g_pos) and i_pos:
        return "stagflation"
    if (not g_pos) and stress_high:
        return "crisis"
    return "slowdown"

scores["regime"] = scores.apply(classify_regime_row, axis=1)
scores["regime"].value_counts(dropna=False)


,count
regime,
stagflation,118
slowdown,110
reflation,88
goldilocks,68
crisis,27
NaN,23


In [8]:

regime_summary = (
    scores.dropna(subset=["regime"])
    .groupby("regime")
    .size()
    .rename("months")
    .to_frame()
)
regime_summary["pct"] = (100 * regime_summary["months"] / regime_summary["months"].sum()).round(1)
regime_summary.sort_values("months", ascending=False)


,months,pct
regime,,
stagflation,118,28.7
slowdown,110,26.8
reflation,88,21.4
goldilocks,68,16.5
crisis,27,6.6


## 5. Préparer un jeu de données de classification supervisée

La cible n'est **pas** le régime à `t+1`.  
Nous cherchons ici à résoudre un problème de **classification du régime courant**.

La variable cible est donc le régime économique du mois `t`, construit à partir des scores macro.
Les variables explicatives sont les indicateurs disponibles au même mois `t`.

Cette approche permet de répondre à deux questions :

1. quelles combinaisons de variables discriminent le mieux les régimes sur l'échantillon d'entraînement ;
2. si l'on applique ce modèle à 2020–2026, la chronologie prédite est-elle économiquement cohérente avec la réalité observée ?

In [9]:
X = pd.concat([features, scores[["growth_score", "inflation_score", "stress_score"]]], axis=1).copy()
y = scores["regime"].rename("target_regime")

dataset = pd.concat([X, y], axis=1).dropna(subset=["target_regime"]).copy()

train_mask = dataset.index <= pd.Timestamp(TRAIN_END)
test_mask = dataset.index >= pd.Timestamp(TEST_START)

train = dataset.loc[train_mask].copy()
test = dataset.loc[test_mask].copy()

X_train = train.drop(columns=["target_regime"])
y_train = train["target_regime"]

X_test = test.drop(columns=["target_regime"])
y_test = test["target_regime"]

train.shape, test.shape

((277, 16), (134, 16))

## 6. Tester plusieurs combinaisons d'indicateurs sur l'échantillon d'entraînement

On ne suppose pas a priori qu'il faut utiliser toutes les variables.

On teste plusieurs familles de spécifications :

- les seuls scores composites ;
- les variables de croissance ;
- les variables d'inflation ;
- les variables de stress ;
- les variables brutes sans scores ;
- l'ensemble complet.

La sélection se fait par validation croisée temporelle sur l'échantillon d'entraînement, avec le score **F1 pondéré**.
L'intérêt est double : identifier les blocs réellement utiles et éviter de surcharger inutilement le modèle.

In [10]:
feature_sets = {
    "scores_seuls": ["growth_score", "inflation_score", "stress_score"],
    "croissance_seule": ["pmi_level", "cli_gap_12m", "slope_us", "cesi_us"],
    "inflation_seule": ["cpi_yoy", "breakeven_us", "breakeven_eu_proxy"],
    "stress_seul": ["vix", "move", "fci_us", "cdx_ig", "itraxx_eur"],
    "variables_brutes": list(features.columns),
    "complet": list(X.columns),
}

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

tscv = TimeSeriesSplit(n_splits=5)

rows = []
for name, cols in feature_sets.items():
    fold_scores = []
    X_train_subset = X_train[cols]
    for tr_idx, val_idx in tscv.split(X_train_subset):
        X_tr = X_train_subset.iloc[tr_idx]
        X_val = X_train_subset.iloc[val_idx]
        y_tr = y_train.iloc[tr_idx]
        y_val = y_train.iloc[val_idx]

        model_cv = make_model()
        model_cv.fit(X_tr, y_tr)
        pred_val = model_cv.predict(X_val)
        fold_scores.append(f1_score(y_val, pred_val, average="weighted"))

    rows.append({
        "specification": name,
        "n_features": len(cols),
        "cv_f1_weighted_mean": np.mean(fold_scores),
        "cv_f1_weighted_std": np.std(fold_scores),
    })

cv_results = pd.DataFrame(rows).sort_values(
    ["cv_f1_weighted_mean", "cv_f1_weighted_std"],
    ascending=[False, True]
).reset_index(drop=True)

cv_results

,specification,n_features,cv_f1_weighted_mean,cv_f1_weighted_std
0,scores_seuls,3,0.533034,0.168052
1,inflation_seule,3,0.296263,0.056014
2,complet,15,0.286327,0.050148
3,variables_brutes,12,0.238437,0.044414
4,croissance_seule,4,0.237087,0.146215
5,stress_seul,5,0.090156,0.088188


In [11]:
best_spec = cv_results.loc[0, "specification"]
best_cols = feature_sets[best_spec]

model = make_model()
model.fit(X_train[best_cols], y_train)

pred_test = pd.Series(model.predict(X_test[best_cols]), index=X_test.index, name="pred_regime")
proba_test = pd.DataFrame(
    model.predict_proba(X_test[best_cols]),
    index=X_test.index,
    columns=model.named_steps["clf"].classes_
)

accuracy = accuracy_score(y_test, pred_test)
f1w = f1_score(y_test, pred_test, average="weighted")

print("Meilleure spécification :", best_spec)
print("Variables retenues :", best_cols)
print("Accuracy test :", round(accuracy, 3))
print("F1 pondéré test :", round(f1w, 3))

Meilleure spécification : scores_seuls
Variables retenues : ['growth_score', 'inflation_score', 'stress_score']
Accuracy test : 0.858
F1 pondéré test : 0.861


In [12]:
print(classification_report(y_test, pred_test))

coef_table = pd.DataFrame(
    model.named_steps["clf"].coef_.T,
    index=best_cols,
    columns=model.named_steps["clf"].classes_
).sort_index()

coef_table

              precision    recall  f1-score   support

      crisis       0.50      1.00      0.67         7
  goldilocks       0.79      0.79      0.79        14
   reflation       1.00      0.93      0.96        27
    slowdown       0.89      0.72      0.80        47
 stagflation       0.88      0.97      0.93        39

    accuracy                           0.86       134
   macro avg       0.81      0.88      0.83       134
weighted avg       0.88      0.86      0.86       134



,crisis,goldilocks,reflation,slowdown,stagflation
growth_score,-2.080751,2.126375,2.425644,-0.392983,-2.078284
inflation_score,-1.866604,-1.975547,2.226702,-1.359978,2.975427
stress_score,2.121141,-1.350247,-0.993886,0.537621,-0.314629


## 7. Comparer les régimes réalisés et classifiés sur l'échantillon de test (2015–2026)

In [13]:
comparison = pd.DataFrame({
    "regime_reconstruit": y_test,
    "regime_classifie_ml": pred_test
}, index=X_test.index)

comparison["correct"] = comparison["regime_reconstruit"] == comparison["regime_classifie_ml"]
comparison.head(24)

,regime_reconstruit,regime_classifie_ml,correct
date,,,
2015-01-30,slowdown,crisis,False
2015-02-27,slowdown,slowdown,True
2015-03-31,slowdown,crisis,False
2015-04-30,slowdown,slowdown,True
2015-05-29,slowdown,slowdown,True
2015-06-30,slowdown,slowdown,True
2015-07-31,slowdown,slowdown,True
2015-08-31,crisis,crisis,True
2015-09-30,crisis,crisis,True


## 8. Validation économique externe sur 2015–2026

Au-delà de la métrique statistique, il faut vérifier si la chronologie prédite est cohérente avec les grands faits macroéconomiques observés.

Lecture économique de référence à utiliser pour l'interprétation :

- **2020** : choc de crise mondiale lié à la pandémie, effondrement de l'activité et stress financier extrême.
- **2021** : reprise très forte, normalisation graduelle, régime de reflation.
- **2022** : choc inflationniste mondial et choc énergétique, resserrement monétaire rapide, épisode proche de la stagflation.
- **2023** : ralentissement de la croissance mais désinflation progressive ; beaucoup d'économies évitent la récession profonde.
- **2024** : croissance modérée mais résiliente, inflation en baisse ; environnement plus proche d'un ralentissement / Goldilocks imparfait selon les zones.
- **2025–début 2026** : croissance encore positive mais plus fragile, inflation moins dominante qu'en 2022, risques liés aux conditions financières et au commerce.

Sources institutionnelles recommandées pour cette comparaison qualitative :
IMF World Economic Outlook (octobre 2020, octobre 2022, octobre 2023, avril 2024, avril 2025) ; OECD Economic Outlook (2023, 2024, 2025) ; bulletins macro de la BCE.

## 8bis. Traduire les régimes en pondérations de portefeuille

Ces allocations ne sont qu'un point de départ.
Elles devront ensuite être estimées ou validées à partir des rendements historiques des classes d'actifs.

L'idée économique est la suivante :

- **crisis** : surpondérer taux / crédit, sous-pondérer actions et matières premières ;
- **goldilocks** : surpondérer actions et devises de portage ;
- **reflation** : surpondérer matières premières et actions, sous-pondérer la duration ;
- **stagflation** : privilégier les actifs réels et réduire les expositions sensibles aux taux.

In [14]:

regime_weights = {
    "goldilocks": {"equity": 0.35, "rates_credit": 0.20, "fx": 0.25, "commodities": 0.20},
    "reflation":  {"equity": 0.30, "rates_credit": 0.15, "fx": 0.20, "commodities": 0.35},
    "stagflation":{"equity": 0.15, "rates_credit": 0.15, "fx": 0.20, "commodities": 0.50},
    "crisis":     {"equity": 0.10, "rates_credit": 0.50, "fx": 0.25, "commodities": 0.15},
    "slowdown":   {"equity": 0.25, "rates_credit": 0.30, "fx": 0.20, "commodities": 0.25},
}

weights_df = pd.DataFrame(regime_weights).T
weights_df



,equity,rates_credit,fx,commodities
goldilocks,0.35,0.20,0.25,0.20
reflation,0.30,0.15,0.20,0.35
stagflation,0.15,0.15,0.20,0.50
crisis,0.10,0.50,0.25,0.15
slowdown,0.25,0.30,0.20,0.25


In [15]:
scores[['regime']].dropna().to_csv('macro_regimes.csv')

## 9. Étape suivante : relier le modèle macro aux rendements d'actifs

Pour terminer le projet d'investissement, il manque encore un jeu de données : les rendements mensuels de vos sleeves de stratégie.

Il faudra idéalement disposer des séries suivantes :

- equity value ;
- equity momentum ;
- rates / credit value ;
- rates / credit momentum ;
- FX value ;
- FX momentum ;
- commodities value ;
- commodities momentum.

Ensuite, vous pourrez comparer trois portefeuilles :

1. le benchmark égal pondéré ;
2. un portefeuille dynamique utilisant les régimes reconstruits ;
3. un portefeuille dynamique réellement investissable utilisant les régimes classifiés par le modèle.

## Remarques pratiques

Ce notebook est volontairement une base propre, pas un modèle final.

Les améliorations les plus importantes seront les suivantes :

- enrichir la recherche de spécifications en testant davantage de combinaisons de variables ;
- comparer la régression logistique à des modèles non linéaires ;
- utiliser une validation en marche avant plus stricte ;
- comparer systématiquement la chronologie prédite 2020–2026 aux grands événements macro documentés ;
- intégrer les rendements d'actifs et calculer Sharpe, drawdown, turnover et coûts de transaction.